# Live exercise: make it robust, worked solution

In [ ]:
import csv
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

here = Path.cwd()
while not (here / "data" / "clean" / "plays.csv").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

PLAYS = "data/clean/plays.csv"

## Given

In [ ]:
def total(numbers):
    """Return the sum of a list of numbers."""
    running = 0
    for number in numbers:
        running += number
    return running


def average(numbers):
    """Return the mean of a list of numbers."""
    return total(numbers) / len(numbers)


def highest(numbers):
    """Return the largest value in a list of numbers."""
    best = numbers[0]
    for number in numbers:
        if number > best:
            best = number
    return best


def load_rows(path):
    """Return the CSV at path as a list of dictionaries, one per row."""
    with open(path, encoding="utf-8") as f:
        return list(csv.DictReader(f))


def minutes_on(rows, device):
    """Return the minutes_played of every play on one device, as numbers."""
    return [float(row["minutes_played"]) for row in rows if row["device"] == device]

## 1. `safe_highest`

`highest([])` fails on its first line, `best = numbers[0]`, with an
`IndexError`: there is no position 0 in an empty list.

In [ ]:
def safe_highest(numbers):
    """Return the largest value in a list, or None if the list is empty."""
    try:
        return highest(numbers)
    except IndexError:
        return None


print(safe_highest([3, 10, 7]))
print(safe_highest([]))

It wraps `highest` rather than changing it. `highest` still crashes on an
empty list, which is right for code that should never see one; code that
expects empty lists calls the safe version. The name tells you which is
which.

## 2. `to_minutes`

In [ ]:
def to_minutes(text):
    """Return messy minutes text like "4.65" or "4.2 min" as a float, or None."""
    text = text.strip()
    if text.endswith(" min"):
        text = text[:-len(" min")]
    try:
        return float(text)
    except ValueError:
        return None


print(to_minutes("4.65"), to_minutes("4.2 min"), to_minutes(""))

Only `float()` is inside the `try`. The strip and the `" min"` check cannot
fail, so they stay outside. **Keep a `try` block as small as the one line
that can raise**: the more you put inside it, the more chance it catches
something you did not mean it to.

In [ ]:
with open("data/messy/plays_messy.csv", encoding="utf-8") as f:
    raw = [row["minutes played "] for row in csv.DictReader(f)]

minutes = [to_minutes(text) for text in raw]
numbers = [m for m in minutes if m is not None]
missing = len(minutes) - len(numbers)

print(f"{len(numbers)} numbers, {missing} None")
assert (len(numbers), missing) == (2129, 94)

2,129 numbers, up from 2,033 with plain `to_number`: the 96 `"4.2 min"`
values are recovered. The 94 left over are all empty strings, and no code
can recover a number nobody wrote down. Those stay missing, and saying so is
the honest answer.

## 3. `summary_line`

In [ ]:
def summary_line(path, device):
    """Return one sentence about the average play on a device, whatever happens."""
    try:
        rows = load_rows(path)
    except FileNotFoundError:
        return f"No file at {path}. Check the spelling, and run from the repository root."

    minutes = minutes_on(rows, device)
    try:
        return (f"{len(minutes)} plays on the {device}, "
                f"{average(minutes):.2f} minutes on average")
    except ZeroDivisionError:
        return f"No plays on the {device}, so there is no average."


print(summary_line(PLAYS, "phone"))
print(summary_line(PLAYS, "radio"))
print(summary_line("data/clean/play.csv", "phone"))

Two `try` blocks, one per thing that can go wrong, each naming exactly one
error. One big `try` around the whole function with `except Exception`
would also "work", and would also report a mistyped column name as "No
file at ...". That is the bare-except lie from the walkthrough, in a more
respectable coat.

## Part 2: the tests

In [ ]:
def test_safe_highest_of_a_normal_list():
    assert safe_highest([3, 10, 7]) == 10


def test_safe_highest_with_only_negatives():
    assert safe_highest([-5, -2, -9]) == -2


def test_safe_highest_of_an_empty_list_is_none():
    assert safe_highest([]) is None


def test_to_minutes_plain_number():
    assert to_minutes("4.65") == 4.65


def test_to_minutes_with_min_on_the_end():
    assert to_minutes("4.2 min") == 4.2


def test_to_minutes_empty_text_is_none():
    assert to_minutes("") is None


def test_to_minutes_words_are_none():
    assert to_minutes("about four") is None


def test_load_rows_missing_file_raises():
    try:
        load_rows("data/clean/no_such_file.csv")
    except FileNotFoundError:
        return
    raise AssertionError("load_rows did not raise FileNotFoundError")


def test_summary_line_for_the_phone():
    assert summary_line(PLAYS, "phone") == "1039 plays on the phone, 4.15 minutes on average"


def test_summary_line_for_a_device_with_no_plays():
    assert summary_line(PLAYS, "radio") == "No plays on the radio, so there is no average."


def test_summary_line_for_a_missing_file():
    assert summary_line("data/clean/play.csv", "phone").startswith("No file at")

In [ ]:
def run_tests(tests):
    """Call every test, report each one, and return how many failed."""
    failed = 0
    for test in tests:
        try:
            test()
            print(f"PASSED  {test.__name__}")
        except AssertionError:
            print(f"FAILED  {test.__name__}")
            failed += 1
    print(f"{len(tests) - failed} passed, {failed} failed")
    return failed


assert run_tests([
    test_safe_highest_of_a_normal_list,
    test_safe_highest_with_only_negatives,
    test_safe_highest_of_an_empty_list_is_none,
    test_to_minutes_plain_number,
    test_to_minutes_with_min_on_the_end,
    test_to_minutes_empty_text_is_none,
    test_to_minutes_words_are_none,
    test_load_rows_missing_file_raises,
    test_summary_line_for_the_phone,
    test_summary_line_for_a_device_with_no_plays,
    test_summary_line_for_a_missing_file,
]) == 0

`test_load_rows_missing_file_raises` is the one worth a second look. It
checks that a function **does** crash, and that pins down a decision:
`load_rows` must never quietly return an empty list for a missing file. If
somebody "helpfully" adds a `try` to it later, this test fails and tells
them why that is a bad idea.

## The file version

The same functions are in `solutions/robust_solved.py`, and the same tests,
with `pytest.raises` for the missing file, are in
`solutions/test_robust_solved.py`. The file version has one extra test,
`test_to_minutes_ignores_surrounding_spaces`, so it reports 12.

In [ ]:
if importlib.util.find_spec("pytest") is None:
    print("pytest is not installed in this Python:  python -m pip install pytest")
else:
    result = subprocess.run(
        [sys.executable, "-m", "pytest", "sessions/session-12/solutions/test_robust_solved.py",
         "-p", "no:cacheprovider"],
        capture_output=True, text=True,
    )
    print(result.stdout)
    assert result.returncode == 0